# 03 - SQL Analysis with DuckDB

## Purpose

This notebook uses DuckDB to run SQL analysis on the cleaned sales dataset.

The goal is to calculate business KPIs and prepare query outputs that can support an executive sales dashboard.

In [1]:
import duckdb
from pathlib import Path

processed_path = Path("../data/processed")
clean_sales_file = processed_path / "clean_sales.csv"

In [2]:
con = duckdb.connect()

In [3]:
con.execute(f"""
CREATE OR REPLACE VIEW clean_sales AS
SELECT *
FROM read_csv(
    '{clean_sales_file}',
    header = true,
    columns = {{
        'invoiceno': 'VARCHAR',
        'stockcode': 'VARCHAR',
        'description': 'VARCHAR',
        'quantity': 'INTEGER',
        'invoicedate': 'TIMESTAMP',
        'unitprice': 'DOUBLE',
        'customerid': 'VARCHAR',
        'country': 'VARCHAR',
        'revenue': 'DOUBLE',
        'transaction_type': 'VARCHAR'
    }}
)
""")

In [4]:
con.execute("""
SELECT *
FROM clean_sales
LIMIT 5
""").df()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,revenue,transaction_type
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,Sale
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Sale
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,Sale
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Sale
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Sale


In [5]:
con.execute("""
DESCRIBE clean_sales""").df()

,column_name,column_type,null,key,default,extra
0,invoiceno,VARCHAR,YES,None,None,None
1,stockcode,VARCHAR,YES,None,None,None
2,description,VARCHAR,YES,None,None,None
3,quantity,INTEGER,YES,None,None,None
4,invoicedate,TIMESTAMP,YES,None,None,None
5,unitprice,DOUBLE,YES,None,None,None
6,customerid,VARCHAR,YES,None,None,None
7,country,VARCHAR,YES,None,None,None
8,revenue,DOUBLE,YES,None,None,None
9,transaction_type,VARCHAR,YES,None,None,None


In [6]:
con.execute("""
SELECT
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoiceno) AS total_orders,
    COUNT(*) AS total_line_items,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(revenue) / COUNT(DISTINCT invoiceno), 2) AS average_order_value
FROM clean_sales
""").df()

,total_revenue,total_orders,total_line_items,total_quantity_sold,average_order_value
0,10631048.74,19959,524877,5572419.0,532.64


In [7]:
con.execute("""
SELECT
    DATE_TRUNC('month', invoicedate::TIMESTAMP) AS sales_month,
    ROUND(SUM(revenue), 2) AS monthly_revenue,
    COUNT(DISTINCT invoiceno) AS monthly_orders,
    SUM(quantity) AS monthly_quantity_sold
FROM clean_sales
GROUP BY sales_month
ORDER BY sales_month
""").df()

,sales_month,monthly_revenue,monthly_orders,monthly_quantity_sold
0,2010-12-01,821452.73,1559,358019.0
1,2011-01-01,689811.61,1086,387099.0
2,2011-02-01,522545.56,1100,282934.0
3,2011-03-01,716215.26,1454,376599.0
4,2011-04-01,536968.49,1246,307953.0
5,2011-05-01,769296.61,1681,395001.0
6,2011-06-01,760547.01,1533,388511.0
7,2011-07-01,718076.12,1475,399693.0
8,2011-08-01,746779.32,1360,421019.0
9,2011-09-01,1056435.19,1837,569573.0


In [8]:
con.execute("""
SELECT
    stockcode,
    description,
    ROUND(SUM(revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity_sold,
    COUNT(DISTINCT invoiceno) AS order_count
FROM clean_sales
GROUP BY stockcode, description
ORDER BY total_revenue DESC
LIMIT 10
""").df()

,stockcode,description,total_revenue,total_quantity_sold,order_count
0,DOT,DOTCOM POSTAGE,206248.77,706.0,706
1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851.0,1988
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995.0,1
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37580.0,2189
4,47566,PARTY BUNTING,99445.23,18283.0,1685
5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371.0,2089
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033.0,247
7,POST,POSTAGE,78101.88,3150.0,1126
8,M,Manual,77750.27,6984.0,289
9,23084,RABBIT NIGHT LIGHT,66870.03,30739.0,994
